In [2]:
import pandas as pd
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

url = "https://exofop.ipac.caltech.edu/tess/download_toi.php?sort=toi&output=csv"
raw_df = pd.read_csv(url)

print(raw_df.columns.tolist())

['TIC ID', 'TOI', 'Previous CTOI', 'Master', 'SG1A', 'SG1B', 'SG2', 'SG3', 'SG4', 'SG5', 'ESM', 'TSM', 'Predicted Mass (M_Earth)', 'Predicted RV Semi-amplitude (m/s)', 'Time Series Observations', 'Spectroscopy Observations', 'Imaging Observations', 'TESS Disposition', 'TFOPWG Disposition', 'TESS Mag', 'TESS Mag err', 'Planet Name', 'Pipeline Signal ID', 'Source', 'Detection', 'RA', 'Dec', 'PM RA (mas/yr)', 'PM RA err (mas/yr)', 'PM Dec (mas/yr)', 'PM Dec err (mas/yr)', 'Epoch (BJD)', 'Epoch (BJD) err', 'Period (days)', 'Period (days) err', 'Duration (hours)', 'Duration (hours) err', 'Depth (mmag)', 'Depth (mmag) err', 'Depth (ppm)', 'Depth (ppm) err', 'Planet Radius (R_Earth)', 'Planet Radius (R_Earth) err', 'Planet Insolation (Earth Flux)', 'Planet Equil Temp (K)', 'Planet SNR', 'Stellar Distance (pc)', 'Stellar Distance (pc) err', 'Stellar Eff Temp (K)', 'Stellar Eff Temp (K) err', 'Stellar log(g) (cm/s^2)', 'Stellar log(g) (cm/s^2) err', 'Stellar Radius (R_Sun)', 'Stellar Radius (R_

In [3]:
print(raw_df["TIC ID"].duplicated().sum())

322


In [ ]:
aux_cols_map = {
    "TIC ID": "TIC_ID",
    "Duration (hours)": "Duration_hrs",
    "Depth (ppm)": "Depth_ppm",
    "Stellar Eff Temp (K)": "Teff_K",
    "Stellar Radius (R_Sun)": "Stellar_Radius",
    "TESS Mag": "TESS_Mag",
}

raw_df = raw_df.drop_duplicates(subset="TIC ID", keep="first")
aux_df = raw_df[list(aux_cols_map.keys())].rename(columns=aux_cols_map)

In [ ]:
labels_df = pd.read_csv("data/processed_labels.csv")
merged_df = labels_df.merge(aux_df, on="TIC_ID", how="left")

assert len(merged_df) == len(labels_df), "merge changed row count — check for TIC_ID duplicates"
print(merged_df.isna().sum())

In [ ]:
merged_df.to_csv("data/processed_labels_v2.csv", index=False)

In [4]:
from collections import defaultdict
import numpy as np

aux_feature_map = {
    "Duration (hours)": "Duration_hrs",
    "Depth (ppm)": "Depth_ppm",
    "Stellar Eff Temp (K)": "Teff_K",
    "Stellar Radius (R_Sun)": "Stellar_Radius",
    "TESS Mag": "TESS_Mag",
}

labels_df = pd.read_csv("data/processed_labels.csv")

raw_by_tic = defaultdict(list)
for _, row in raw_df.iterrows():
    raw_by_tic[row["TIC ID"]].append(row)

matched = []
for _, row in labels_df.iterrows():
    candidates = raw_by_tic.get(row["TIC_ID"], [])
    if not candidates:
        matched.append({v: np.nan for v in aux_feature_map.values()})
        continue
    if len(candidates) > 1:
        target = row["Period (days)"]
        chosen = min(candidates, key=lambda r: abs(r["Period (days)"] - target))
    else:
        chosen = candidates[0]
    matched.append({v: chosen[k] for k, v in aux_feature_map.items()})

aux_df = pd.DataFrame(matched)
merged_df = pd.concat([labels_df.reset_index(drop=True), aux_df], axis=1)

print(merged_df.isna().sum())

TIC_ID              0
final_label         0
Period (days)       0
Epoch (BJD)         0
Duration_hrs        1
Depth_ppm           0
Teff_K             66
Stellar_Radius    232
TESS_Mag            0
dtype: int64


In [5]:
merged_df.to_csv("data/processed_labels_v2.csv", index=False)